# Step 1: Import Relevant LIbraries

In [42]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
#joblib



## STEP 2: Load the cleaned dataset

In [43]:

cleaned_df = pd.read_csv("/home/nyaugenya/dev/personal/AdvanceDataScience/Water Pump Functionality Prediction/data/cleaned_water_pump_data.csv")
cleaned_df.head()

,amount_tsh,gps_height,region,basin,region_code,construction_year,date_recorded,district_code,population,public_meeting,...,extraction_type_class,management_group,payment_type,water_quality,quantity_group,source_type,waterpoint_type_group,status_group,month_recorded,age
0,6000.0,1390,Iringa,Lake Nyasa,11,1970-01-01 00:00:00.000001999,2011-03-14,5,109,True,...,gravity,user-group,annually,soft,enough,spring,communal standpipe,functional,3,41
1,25.0,686,Manyara,Pangani,21,1970-01-01 00:00:00.000002009,2013-02-25,4,250,True,...,gravity,user-group,per bucket,soft,enough,dam,communal standpipe,functional,2,43
2,0.0,263,Mtwara,Ruvuma/Southern Coast,90,1970-01-01 00:00:00.000001986,2013-01-28,63,58,True,...,submersible,user-group,never pay,soft,dry,borehole,communal standpipe,non functional,1,43
3,20.0,0,Tanga,Pangani,4,1970-01-01 00:00:00.000002009,2011-03-13,8,1,True,...,submersible,user-group,per bucket,salty,enough,other,communal standpipe,functional,3,41
4,0.0,0,Shinyanga,Internal,17,1970-01-01 00:00:00.000000000,2012-10-01,3,0,True,...,handpump,user-group,never pay,soft,enough,borehole,hand pump,non functional,10,42


## STEP 3: SPlit the features and the targets, then split into training and test sets


### SPLIT the target from the features

In [44]:
X = cleaned_df.drop(columns=['status_group']) # write your variable as capital letter if is a 2D array 
y = cleaned_df['status_group'] # and a small letter if is a 1D array

X.shape

(49840, 21)

In [45]:
X.shape

(49840, 21)

In [46]:
y.shape

(49840,)

# split into train and test

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y,random_state=42) # stratify=y ensures that the class distribution is preserved(maintained) in both the training and testing sets

In [48]:
((cleaned_df['status_group'].value_counts()) / len(cleaned_df)) * 100

status_group
functional                 54.935795
non functional             37.702648
functional needs repair     7.361557
Name: count, dtype: float64

In [49]:
# Encode the target column coz many machine learning algorithms require the target variable to be in numerical format. We can use label encoding to convert the categorical labels into numerical values.
# _ common_types_of_encoders_
# Ordinal Encoder: This encoder is used when the categorical variable has an inherent order or ranking. It assigns a unique integer to each category based on their order. For example, if we have a feature "Education Level" with categories "High School," "Bachelor's," and "Master's," we can assign 0 to "High School," 1 to "Bachelor's," and 2 to "Master's."
# One-Hot Encoder: This encoder is used when the categorical variable does not have any inherent order. It creates binary columns for each category, indicating the presence or absence of that category. For example, if we have a feature "Color" with categories "Red," "Green," and "Blue," one-hot encoding would create three binary columns: "Color_Red," "Color_Green," and "Color_Blue," where each column would have a value of 1 if the corresponding color is present and 0 otherwise.
# Label Encoder: This encoder is used when the categorical variable is the target variable (the variable we want to predict). It assigns a unique integer to each category. For example, if we have a target variable "Animal" with categories "Dog," "Cat," and "Bird," label encoding would assign 0 to "Dog," 1 to "Cat," and 2 to "Bird." This is useful for algorithms that require numerical labels for classification tasks
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

## STEP 4: Building the preprocessing pipeline
# Why use sklearn's pipeline?
Using sklearn's pipeline is beneficial for several reasons:
- it bundles the data preprocessing and model training into a single ,cohesive object making your machine learning workflows cleaner,safer and production ready
- It helps to streamline the preprocessing and modeling steps, making the code more readable and maintainable.
- It ensures that the same preprocessing steps are applied to both the training and testing data, reducing the risk of data leakage.
- It allows for easy hyperparameter tuning and cross-validation, as the entire pipeline can be treated as a single estimator.
- Instead of manually tracking and multiple `,fit()` and `.transform()` methods across your training and tsting sets, apipeline allows you to execute the entire sequence with a single command

In [50]:
# Numerical Columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('month_recorded') # remove month_recorded from numerical_cols list since it is a categorical variable
print("Numerical Columns:", numerical_cols)

# Categorical Columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
categorical_cols.append('month_recorded') # add month_recorded to categorical_cols list since it is a categorical variable
print("Categorical Columns:", categorical_cols)



Numerical Columns: ['amount_tsh', 'gps_height', 'region_code', 'district_code', 'population', 'age']
Categorical Columns: ['region', 'basin', 'construction_year', 'date_recorded', 'scheme_management', 'extraction_type_class', 'management_group', 'payment_type', 'water_quality', 'quantity_group', 'source_type', 'waterpoint_type_group', 'month_recorded']


/tmp/ipykernel_102018/576661846.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()


### 4.1: NUMERICAL PIPELINE

In [51]:
### 4.1 Numerical Pipeline
# Create a numerical pipeline that imputes missing values with the median and scales the features using StandardScaler. The pipeline consists of two steps: 'imputer' and 'scaler'. The 'imputer' step uses SimpleImputer to fill in missing values with the median of each feature, and the 'scaler' step uses StandardScaler to standardize the features by removing the mean and scaling to unit variance. This pipeline is applied to the numerical columns of the dataset.
numerical_pipeline = Pipeline(steps=[
    # ('imputer', SimpleImputer(strategy='median')), advised to clean your data before training the model
    ('scaler', StandardScaler())
])

### 4.2 CATEGORICAL PIPELINE

In [52]:
### 4.`2 Categorical Pipeline
# Create a categorical pipeline that imputes missing values with the most frequent value and applies one-hot encoding to the categorical features. The pipeline consists of two steps: 'imputer' and 'onehot'. The 'imputer' step uses SimpleImputer to fill in missing values with the most frequent value of each feature, and the 'onehot' step uses OneHotEncoder to convert categorical features into binary columns. The handle_unknown='ignore' parameter in OneHotEncoder ensures that any unknown categories encountered during transformation are ignored, preventing errors during model training and prediction. This pipeline is applied to the categorical columns of the dataset.`

#handle_unknown='ignore' parameter in OneHotEncoder ensures that any unknown categories encountered during transformation are ignored, preventing errors during model training and prediction. This pipeline is applied to the categorical columns of the dataset.
categorical_pipeline = Pipeline(steps=[
    # ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

### Combine the numerical and categorical pipelines using ColumnTransformer

In [53]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)])

## STEP 5 : Train all possible models and pick the best performing model

In [54]:
pipeline= {'LogisticRegression': Pipeline([('preprocessor', preprocessor),('model', LogisticRegression(max_iter=1000))]),
           'KNN': Pipeline([('preprocessor', preprocessor),('model', KNeighborsClassifier(n_neighbors=7))]),
           'Support Vector Machine': Pipeline([('preprocessor', preprocessor),('model', SVC())]),
           'Decision Tree': Pipeline([('preprocessor', preprocessor),('model', DecisionTreeClassifier())]),
           'Random Forest': Pipeline([('preprocessor', preprocessor),('model', RandomForestClassifier())]),
           'Gradient Boosting': Pipeline([('preprocessor', preprocessor),('model', GradientBoostingClassifier())]),
           'XGBoost': Pipeline([('preprocessor', preprocessor),('model', XGBClassifier())])         
           }

for model_name, model_pipeline in pipeline.items():
    model_pipeline.fit(X_train, y_train_encoded)
    y_pred = model_pipeline.predict(X_test)
    print(f"{'-'*40} Model: {model_name}   {'-'*40}")
    print("Classification Report: \n",classification_report(y_test_encoded, y_pred, target_names=le.classes_))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test_encoded, y_pred))
    print("Accuracy Score:", accuracy_score(y_test_encoded, y_pred))
    print("\n")

---------------------------------------- Model: LogisticRegression   ----------------------------------------
Classification Report: 
                          precision    recall  f1-score   support

             functional       0.72      0.90      0.80      8214
functional needs repair       0.52      0.11      0.18      1101
         non functional       0.79      0.64      0.70      5637

               accuracy                           0.74     14952
              macro avg       0.68      0.55      0.56     14952
           weighted avg       0.73      0.74      0.72     14952

Confusion Matrix:
[[7359   60  795]
 [ 808  116  177]
 [1994   47 3596]]
Accuracy Score: 0.7404360620652756


---------------------------------------- Model: KNN   ----------------------------------------
Classification Report: 
                          precision    recall  f1-score   support

             functional       0.77      0.87      0.81      8214
functional needs repair       0.51      0.26  

#### Common Evaluation metrics for evaluation
_i)Confusion Matrix - it's a simple table that compare the model predicted against what actually happenned
|               |predicted Positive|Predicted Negative|
|---------------|------------------|------------------|
|Actual Positive|True Postive(TP)  |False Negative(FN)|
|Actual Negative|False Postive(FP) |True Negative(TN) |

TP: the model predicted positive and it was actually positive
TN: the model predicted negative and it was actually negative
FP: the model predicted postive and it was actually negative(false alarm)
FN: the model predicted negative and it was actually positive(amis)

_ii). Accuracy__
Accuracy tells you the percentage of preditions the model got right,of all the predictions made

$Accuracy = \frac{TP+TN}{TP+TN+FP+FN}$
Accuracy is easy to understand but it can be misleading when classes are imbalanced

_iii). Precision_
Precision answer the question of everything the model labelled as positive ,how many were actually positive?
$Precision = \frac{TP}{TP+FP}$
High precistion means the model dos not raise the false alarm
_Use it when_ False postive are costly or disruptive eg  detecting spam emails , fraud
_Avoid it when_ Missing a positive case is far more constly than a false alarm

_iv). Recall/Sensitivity
Recall naswers the quetsuons of all the actual positives,how many did the model correctly find?
$Recall = \frac{TP}{TP+FN} 
High Recall means the model rarely misses a true positive: This matters when missing a positive case is costly example failing to detect a disease. eg medical test
_Use recall when_ missing a positive case is dangerous or expensive eg missing cancer diagnosis in a screening test

_v). F1 Score__
The F1 Score combines both precision and recall into a single number using their harmonic mean , so it is usefull whennyou want a balanced score

$F1 Score =  2 * \frac{(Precision * Recall)}{(Precision + Recall)}$

The F1-Score is especially helpful when the datasets are imbalanced and accuracy alone is not trustworthy

_vi). Macro F1 Score__
We use it when there are more than two classess for example classifying water pump functionality status status i.e functional, non-functional needs repair.
$Macro F2  = \frac{\text(F1 for class 1 + F1 for class 2 + ...F1 for class n)}{n}$

## STEP 6: Create and Train the model

In [55]:
full_pipeline= Pipeline([('preprocessor',preprocessor),('model',RandomForestClassifier(random_state=42))])

## Train the model

In [56]:
full_pipeline.fit(X_test, y_test_encoded)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](21,)","['amount_tsh','gps_height','region',...,'waterpoint_type_group', 'month_recorded','age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,21
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

## STEP 7: Hyper-Parameter Tuning

- Hyper-parameter tuning is the practise of indentifying ans selecting optimal hyperparameters for use in training machine learning models.
__Cross Validation(CV)__
 Cross Validation(CV)'
| | |
|-|-|

Round 1:
|Test 1 | | |
|-------|-|-|

Round 2:
|Test 2 | | |
|-------|-|-|

Round 3:
|Test 3 | | |
|-------|-|-|

Hyper parameters use in Random Forest:
max_depth - the maximum depth of the tree
n_estimators - Number of trees
min_samples_split - The minimum number of samples before a split happens
min_samples_leaf - The minimum number of samples required to be a leaf node


_i) GridSearchCV__
Grid search evaluates the model againmst every possible combination of predefined hyperparameter values.

```python
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100,200,300,400,500],
    'max_depth': [5,10,20,25],
    'min_samples_split': [2,5,10,25]
}

grid_search=GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1 # no of cpu calls
)

grid_search.fit(X_train,y_train_encoded)

# viewing of the results
print("Best Hyperparameters",grid_search.best_params_) # 500,15,10
print("Best Score",grid_search.best_score_) 
# with the results of best hyperparameters i.e 500,15,10
# full_pipeline= Pipeline([('preprocessor',preprocessor),('model',RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_split=10,random_state=42))])
```
_Limitation
- Extreme computational cost
- Impractical
- Time consuming

_ii). RandomizedCV__
Instead of trying every combinations,it randomly samples from defined domain space,which is more efficient for large scale search

```python
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'n_estimators': [100,200,300,400,500],
    'max_depth': [5,10,20,25],
    'min_samples_split': [2,5,10,25],
    "criterion": ["gini", "entropy"]
}

grid_search=RandomizedSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1 # no of cpu calls
)

grid_search.fit(X_train,y_train_encoded)

# viewing of the results
print("Best Hyperparameters",grid_search.best_params_) # 500,15,10
print("Best Score",grid_search.best_score_) 
# with the results of best hyperparameters i.e 500,15,10
# full_pipeline= Pipeline([('preprocessor',preprocessor),('model',RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_split=10,random_state=42))])
```




FINAL STEP: Save the model and target labels

In [57]:
# save the ml model to a file using joblib. The model is saved in the '../models/' directory with the filename 'water_pump_model.joblib'. This allows for easy loading and reuse of the trained model in future applications without needing to retrain it.
import joblib
joblib.dump(full_pipeline, '../models/water_pump_model.joblib'
            )
# save the encoder to a file using joblib. The encoder is saved in the '../models/' directory with the filename 'label_encoder.joblib'. This allows for easy loading and reuse of the trained encoder in future applications without needing to retrain it.
joblib.dump(le, '../models/label_encoder.joblib')

['../models/label_encoder.joblib']